# Perf and Scaling

Demonstrates memory, FLOPs/MACs, streaming capture knobs, and stats aggregation names.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.log_forward_pass(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.log_forward_pass(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.log_forward_pass(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "ModelLog": log,
    "LayerLog": layer_log,
    "LayerPassLog": layer_pass,
    "ModuleLog": module_log,
    "ModulePassLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnPassLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Perf and Scaling: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.grad.autograd_saved_bytes",
    "tl.grad.grad_fn_memory_cost",
    "tl.stats.Aggregator",
    "tl.stats.Covariance",
    "tl.stats.Mean",
    "tl.stats.PCA",
    "tl.stats.Quantile",
    "tl.stats.StreamingStat",
    "tl.stats.TopK",
    "tl.stats.aggregate",
]

streaming_log = tl.log_forward_pass(
    tiny_model(seed=8).eval(), torch.randn(1, 4), layers_to_save=["linear_1_1"], vis_opt="none"
)
pretty_print_fields(
    layer_pass,
    [
        "flops_forward",
        "flops_backward",
        "macs_forward",
        "macs_backward",
        "autograd_saved_bytes",
        "bytes_peak_at_call",
    ],
)
print("selective saved", streaming_log.layers_with_saved_activations)
print("stats exports", [name for name in dir(tl.stats) if not name.startswith("_")][:8])

## Perf and Scaling coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.grad.autograd_saved_bytes",
    "tl.grad.grad_fn_memory_cost",
    "tl.stats.Aggregator",
    "tl.stats.Covariance",
    "tl.stats.Mean",
    "tl.stats.PCA",
    "tl.stats.Quantile",
    "tl.stats.StreamingStat",
    "tl.stats.TopK",
    "tl.stats.aggregate",
]

audit_touch_items("Perf and Scaling coverage cluster 1", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.grad.autograd_saved_bytes",
    "tl.grad.grad_fn_memory_cost",
    "tl.stats.Aggregator",
    "tl.stats.Covariance",
    "tl.stats.Mean",
    "tl.stats.PCA",
    "tl.stats.Quantile",
    "tl.stats.StreamingStat",
    "tl.stats.TopK",
    "tl.stats.aggregate",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")